# 05 - Validations

Collected validation checks for the Marathos pipeline.
Each section validates a specific layer or step in the data flow.

Goal is to have one place to verify that the pipeline behaves as expected -
useful when something changes upstream and we need to confirm nothing broke.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F

from utils.constants import (
    BRONZE_RESULTS_TABLE,
    SILVER_RESULTS_OBT,
    RAW_CSV_PATH,
)
from utils.io_helpers import read_table, read_csv_from_volume
from utils.cleaning import (
    split_event_distance,
    split_performance,
    time_to_seconds,
    add_event_type,
    add_validity_flag,
    clean_rows,
)
from utils.id_generation import add_event_id, add_athlete_id

bronze_df = read_table(spark, BRONZE_RESULTS_TABLE)
silver_df = read_table(spark, SILVER_RESULTS_OBT)

## Bronze validations

Bronze should be an exact copy of the source CSV. We verify rowcount matches
and that no unexpected schema changes happened during the write.

In [0]:
# Bronze should have the same number of rows as the source CSV
csv_df = read_csv_from_volume(spark, RAW_CSV_PATH)

csv_rows = csv_df.count()
bronze_rows = bronze_df.count()

print(f"CSV rows:    {csv_rows:,}")
print(f"Bronze rows: {bronze_rows:,}")
print(f"Match: {csv_rows == bronze_rows}")

### Bronze schema check

The 13 expected columns from the source CSV should be present.

In [0]:
expected_columns = {
    "Year of event", "Event dates", "Event name", "Event distance/length",
    "Event number of finishers", "Athlete performance", "Athlete club",
    "Athlete country", "Athlete year of birth", "Athlete gender",
    "Athlete age category", "Athlete average speed", "Athlete ID",
}

actual_columns = set(bronze_df.columns)

missing = expected_columns - actual_columns
extra = actual_columns - expected_columns

print(f"Expected: {len(expected_columns)} columns")
print(f"Actual:   {len(actual_columns)} columns")
print(f"Missing:  {missing if missing else 'none'}")
print(f"Extra:    {extra if extra else 'none'}")

## Silver parser validations

These check that each parser function gives the expected output on known inputs.
Useful for catching regressions if the cleaning logic gets changed later.

### split_event_distance

`50km` should give value 50.0 and unit "km".
`32.4mi` should give value 32.4 and unit "mi".

In [0]:
# Build a tiny test dataframe with known inputs
test_data = [("50km",), ("32.4mi",), ("24h",), ("6d",)]
test_df = spark.createDataFrame(test_data, ["Event distance/length"])

result = split_event_distance(test_df)
result.show(truncate=False)

### split_performance

Distance values like `50.000 km` should populate `perf_distance`.
Time values like `6:43:49 h` should populate `perf_seconds`.

In [0]:
test_data = [("50.000 km",), ("127.5 km",), ("6:43:49 h",), ("0:30:00 h",)]
test_df = spark.createDataFrame(test_data, ["Athlete performance"])

result = split_performance(test_df)
result.show(truncate=False)

### time_to_seconds with day-format

Performance over a day boundary like `2d 02:17:00 h` should still parse correctly.
2 days + 2 hours + 17 minutes + 0 seconds = 181020 seconds.

In [0]:
test_data = [("2d 02:17:00 h",), ("9d 23:51:20 h",)]
test_df = spark.createDataFrame(test_data, ["Athlete performance"])

result = test_df.withColumn(
    "seconds",
    time_to_seconds(F.col("Athlete performance"))
)
result.show(truncate=False)

## Silver pipeline validations

These check the silver OBT as a whole - that the row counts match expectations,
that the cleaning removed what we think it removed, and that the NULL pattern
matches the event_type design.

### Validity flag distribution

Re-run the validity check on bronze to see how many rows were considered valid
vs invalid. Should match what we saw when building silver.

In [0]:
# Re-build the validity flag from bronze to see the distribution
bronze_with_flag = (
    bronze_df
    .transform(split_event_distance)
    .transform(split_performance)
    .transform(add_validity_flag)
)

bronze_with_flag.groupBy("is_valid").count().show()

### Row count audit trail

How many rows did we lose from bronze to silver, and where did they go?
This is the kind of question stakeholders ask, so it's worth being able to answer.

In [0]:
# Total bronze rows
bronze_count = bronze_df.count()

# After validity filtering (drops invalid event/performance combos + NULL perf rows)
after_validity = bronze_with_flag.filter(F.col("is_valid") == True).count()

# After age filtering (drops rows with missing or implausible birth years)
after_age = (
    bronze_with_flag
    .filter(F.col("is_valid") == True)
    .withColumn("age_at_event", F.col("Year of event") - F.col("Athlete year of birth"))
    .filter((F.col("age_at_event") >= 5) & (F.col("age_at_event") <= 100))
    .count()
)

silver_count = silver_df.count()

print(f"Bronze rows:              {bronze_count:,}")
print(f"After validity filter:    {after_validity:,}  (dropped {bronze_count - after_validity:,})")
print(f"After age filter:         {after_age:,}  (dropped {after_validity - after_age:,})")
print(f"Silver rows:              {silver_count:,}")
print(f"Total dropped:            {bronze_count - silver_count:,} ({(bronze_count - silver_count) / bronze_count * 100:.2f}%)")

### NULL pattern matches event_type

By design, distance events should have NULL `performance_distance` (we recorded their time).
Time events and multi_day events should have NULL `performance_seconds` (we recorded their distance).

If these numbers don't add up, something is wrong with the split logic.

In [0]:
# Number of rows per event_type
event_type_counts = silver_df.groupBy("event_type").count().collect()
event_type_counts_dict = {row["event_type"]: row["count"] for row in event_type_counts}

# Number of NULLs in the performance columns
null_perf_seconds = silver_df.filter(F.col("performance_seconds").isNull()).count()
null_perf_distance = silver_df.filter(F.col("performance_distance").isNull()).count()

distance_count = event_type_counts_dict.get("distance", 0)
time_count = event_type_counts_dict.get("time", 0)
multi_day_count = event_type_counts_dict.get("multi_day", 0)

print(f"Distance events:                 {distance_count:,}")
print(f"NULLs in performance_distance:   {null_perf_distance:,}")
print(f"Match (should be equal):         {distance_count == null_perf_distance}")
print()
print(f"Time + multi_day events:         {time_count + multi_day_count:,}")
print(f"NULLs in performance_seconds:    {null_perf_seconds:,}")
print(f"Match (should be equal):         {time_count + multi_day_count == null_perf_seconds}")

## Country enrichment check (Bonus 1)

After joining dim_country into the OBT, every row should have a country_name and
continent. If any are NULL, it means a country code in silver doesn't exist in
dim_country and the mapping needs to be extended.

In [0]:
# Check that the country join covered every row
missing_country = silver_df.filter(F.col("country_name").isNull())
missing_count = missing_country.count()

print(f"Rows with NULL country_name: {missing_count:,}")

if missing_count > 0:
    print("\nCountry codes that didn't match dim_country:")
    missing_country.groupBy("athlete_country").count().orderBy(F.col("count").desc()).show()

## Date parsing NULL count (Bonus 5)

A small number of rows in the source have invalid dates like "00.03.1998"
(day zero doesn't exist). We use try_to_date to tolerate these instead of
crashing, which gives us NULL start_date and end_date for those rows.

Expected: a small fraction of total rows (<0.1%). If it grows significantly,
something else is wrong with the parsing.

In [0]:
from utils.constants import SILVER_RESULTS_OBT
silver_df = read_table(spark, SILVER_RESULTS_OBT)

null_start = silver_df.filter(F.col("start_date").isNull()).count()
null_end = silver_df.filter(F.col("end_date").isNull()).count()
total = silver_df.count()

print(f"Rows with NULL start_date: {null_start:,} ({null_start/total*100:.3f}%)")
print(f"Rows with NULL end_date:   {null_end:,} ({null_end/total*100:.3f}%)")
print(f"Total rows:                {total:,}")

## Gold validations

Cross-layer checks that verify the fact table's foreign keys actually
exist in their dimension tables. Any "orphan" row in fact means a broken
join later in the dashboard or Genie queries.

In [0]:
from utils.constants import (
    GOLD_FCT_RESULTS,
    GOLD_DIM_EVENT,
    GOLD_DIM_ATHLETE,
    GOLD_DIM_COUNTRY,
    GOLD_DIM_DATE,
)

fct_df = read_table(spark, GOLD_FCT_RESULTS)
dim_event_df = read_table(spark, GOLD_DIM_EVENT)
dim_athlete_df = read_table(spark, GOLD_DIM_ATHLETE)
dim_country_df = read_table(spark, GOLD_DIM_COUNTRY)
dim_date_df = read_table(spark, GOLD_DIM_DATE)

### Foreign key integrity

For every fact row, the referenced dimension key should exist in the dimension table.
Anti-join finds rows where the join fails - we want zero of those.

In [0]:
# event_id orphans
orphan_events = fct_df.join(dim_event_df, "event_id", "left_anti").count()

# athlete_id orphans
orphan_athletes = fct_df.join(dim_athlete_df, "athlete_id", "left_anti").count()

# country_code orphans
orphan_countries = fct_df.join(dim_country_df, "country_code", "left_anti").count()

# date orphans - start_date_id and end_date_id (allow NULL because of try_to_date)
orphan_start_dates = fct_df.filter(F.col("start_date_id").isNotNull()).join(
    dim_date_df.select(F.col("date_id").alias("start_date_id")),
    "start_date_id",
    "left_anti"
).count()

orphan_end_dates = fct_df.filter(F.col("end_date_id").isNotNull()).join(
    dim_date_df.select(F.col("date_id").alias("end_date_id")),
    "end_date_id",
    "left_anti"
).count()

print(f"Orphan event_ids:    {orphan_events:,}")
print(f"Orphan athlete_ids:  {orphan_athletes:,}")
print(f"Orphan country_codes: {orphan_countries:,}")
print(f"Orphan start_date_ids: {orphan_start_dates:,}")
print(f"Orphan end_date_ids:   {orphan_end_dates:,}")